# 14 — Evaluasi Relevansi NLP Multi-Label dan Benchmark Independen

> **Ketentuan Evaluasi Proyek**:
- Gunakan evaluasi relevansi independen dan metrik multi-label yang sesuai sebelum memilih model terbaik.
- **Jangan menggunakan skor similarity atau label pemetaan otomatis lama sebagai bukti akurasi**.
- Evaluasi independen mencakup 30 buku uji (20 katalog + 10 buku baru unseen) lintas 8 domain pekerjaan BPJS (Kesehatan, IT, Data, Audit, Risiko, Hukum, SDM, Keuangan) terhadap seluruh 67 kompetensi dengan derajat relevansi bertingkat (0–3).

## A. Setup Lingkungan dan Pemuatan Benchmark Independen

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent
assert (ROOT / 'data' / 'dataset_buku_2000.csv').exists()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

OUT = ROOT / 'outputs'
FIG = ROOT / 'figures'

# Muat hasil benchmark independen yang telah dihitung
eval_path = OUT / 'evaluasi_metrik_perbandingan.csv'
assert eval_path.exists(), 'File evaluasi_metrik_perbandingan.csv belum ada'
eval_df = pd.read_csv(eval_path)
print('Tabel Metrik Evaluasi Independen Multi-Label:')
display(eval_df)

## B. Analisis Metrik Evaluasi Relevansi (Multi-Label Ranking)

Metrik yang dievaluasi mencakup:
- **NDCG@3 & NDCG@5**: Mengukur kualitas ranking kompetensi dengan bobot relevansi bergradasi (0–3).
- **Precision@1, @3, @5**: Proporsi kompetensi yang benar-benar relevan pada Top-K.
- **MAP@5**: Mean Average Precision untuk presisi urutan rekomendasi.
- **Recall@5**: Cakupan kompetensi relevan yang berhasil direkomendasikan pada Top-5.
- **Hit Rate@3 & @5**: Persentase buku yang mendapatkan minimal satu rekomendasi relevan.
- **Rata-rata Latensi (ms)**: Efisiensi waktu komputasi inferensi per query buku.

In [ ]:
# Format tampilan tabel metrik untuk presentasi
display_cols = ['model', 'ndcg@5', 'ndcg@3', 'precision@1', 'precision@3', 'precision@5', 'recall@5', 'map@5', 'avg_latency_ms']
metric_table = eval_df[display_cols].copy()
metric_table.columns = ['Model NLP', 'NDCG@5', 'NDCG@3', 'P@1', 'P@3', 'P@5', 'Recall@5', 'MAP@5', 'Latensi (ms)']
display(metric_table.sort_values('NDCG@5', ascending=False).reset_index(drop=True))

# Simpan tabel kualitas multilabel untuk notebook 15
eval_df.to_csv(OUT / '14_kualitas_multilabel.csv', index=False)

## C. Visualisasi Perbandingan Metrik Relevansi

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Bar Chart NDCG@5
models_sorted = eval_df.sort_values('ndcg@5', ascending=True)
axes[0].barh(models_sorted['model'], models_sorted['ndcg@5'], color='cornflowerblue')
axes[0].set_title('NDCG@5 (Ranking Relevansi Multi-Label)')
axes[0].set_xlabel('NDCG@5')
axes[0].set_xlim(0, 1.0)
for i, v in enumerate(models_sorted['ndcg@5']):
    axes[0].text(v + 0.02, i, f'{v:.3f}', va='center')

# 2. Bar Chart Precision@1
models_p1 = eval_df.sort_values('precision@1', ascending=True)
axes[1].barh(models_p1['model'], models_p1['precision@1'], color='mediumseagreen')
axes[1].set_title('Precision@1 (Akurasi Top Match)')
axes[1].set_xlabel('Precision@1')
axes[1].set_xlim(0, 1.0)
for i, v in enumerate(models_p1['precision@1']):
    axes[1].text(v + 0.02, i, f'{v:.3f}', va='center')

# 3. Bar Chart MAP@5
models_map = eval_df.sort_values('map@5', ascending=True)
axes[2].barh(models_map['model'], models_map['map@5'], color='coral')
axes[2].set_title('MAP@5 (Mean Average Precision)')
axes[2].set_xlabel('MAP@5')
axes[2].set_xlim(0, 1.0)
for i, v in enumerate(models_map['map@5']):
    axes[2].text(v + 0.02, i, f'{v:.3f}', va='center')

plt.tight_layout()
plt.savefig(FIG / '14_evaluasi_metrik_perbandingan.png', dpi=150)
plt.show()

## D. Kesimpulan Tahap Evaluasi

1. **BGE Reranker v2 M3** menghasilkan relevansi tertinggi (NDCG@5 = 0.7249, P@1 = 86.67%, MAP@5 = 0.5925).
2. **BGE-M3** merupakan Bi-Encoder terbaik (NDCG@5 = 0.7095, P@1 = 86.67%, MAP@5 = 0.5695) dengan latensi 3.83 ms.
3. **Multilingual E5** menunjukkan performa kompetitif (NDCG@5 = 0.7047, P@1 = 86.67%).
4. **TF-IDF** merupakan baseline leksikal terunggul (NDCG@5 = 0.6780, P@1 = 83.33%) dengan latensi super cepat (0.34 ms).
5. **BM25** memiliki NDCG@5 = 0.6507, namun sensitif terhadap vocabulary mismatch.
6. **FastText** memiliki NDCG@5 terendah (0.3476) karena averaging vektor kata mengaburkan informasi domain khusus.